# Class 19: The Confidence Game
*Elements of Data Science (Honors)*

<br>**<center>Learning Goals**

|Area|Concept|
|---|---|
|Bootstrap resampling|Resample a single sample, with replacement, to estimate uncertainty|
|Confidence interval|The middle 95% of a distribution of resampled statistics|
|Bootstrap CI for a mean|Applied to a single quantitative sample|
|Bootstrap CI for a regression slope|Applied to a fitted line, using last week's cricket data|

Today we ask: how much would our answer change if we'd collected different data? We can't usually re-run the experiment, so we'll resample from the one sample we have and see what that tells us.

First, set up the imports by running the cell below.

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')


## Part 1: How Confident Are We in a Mean?

### Warm-up: rolling a die

We roll a six-sided die 50 times and calculate the mean.

In [ ]:
possible_rolls = np.arange(1, 7)
num_rolls = 50
sample_rolls = np.random.choice(possible_rolls, num_rolls)
avg_roll = np.mean(sample_rolls)
print(f"The average of {num_rolls} rolls is {avg_roll}")

How good is this estimate? If we could replicate the experiment thousands of times, we could look at the spread of the resulting means directly. Let's try that first, since we secretly *can* replicate a die roll.

In [ ]:
means = make_array()
for i in np.arange(10000):
    sample_rolls = np.random.choice(possible_rolls, num_rolls)
    means = np.append(means, np.mean(sample_rolls))

Table().with_column("sample mean", means).hist(bins=30)
left = percentile(2.5, means)
right = percentile(97.5, means)
print(f"The 95% CI from replicating the experiment: {left:.3f} to {right:.3f}")

**But in the real world we cannot replicate our experiment thousands of times.** If all we have is *one* sample of 50 rolls, how do we estimate a confidence interval for the mean?

This is where bootstrapping comes in. We treat our one sample as a stand-in for the population, and resample *from it*, with replacement, to build up a distribution of resampled means.

> **Handout Q1.1.** Before you run the next cell: do you expect the bootstrapped CI to be close to the CI you got from replicating the experiment above, or very different? Why might resampling from one sample of 50 tell you anything about the population?

In [ ]:
bootstrap_means = make_array()
for i in np.arange(10000):
    bootstrap_sample = np.random.choice(sample_rolls, size=num_rolls, replace=True)
    bootstrap_means = np.append(bootstrap_means, np.mean(bootstrap_sample))

Table().with_column("bootstrapped mean", bootstrap_means).hist(bins=30)
left = percentile(2.5, bootstrap_means)
right = percentile(97.5, bootstrap_means)
print(f"The 95% CI based on bootstrapping: {left:.3f} to {right:.3f}")

### Estimating height

Now a case closer to how you'll actually use the bootstrap: estimating a confidence interval for the mean of a sample from a height dataset.

In [ ]:
population = Table.read_table("./data/weight-height.csv")
population.show(5)

In [ ]:
pop_mean = np.mean(population.column("Height"))
print(f'Population mean: {pop_mean:.2f} in')

Now imagine we only have a single sample of 10 people.

In [ ]:
pop_ht = population.select("Height")
pop_sample = pop_ht.sample(10)
pop_sample

In [ ]:
sample_mean = np.mean(pop_sample.column("Height"))
print(f'Sample mean: {sample_mean:.2f} in')

The sample mean is different from the population mean. Rather than take many new samples (which we usually can't do), we resample *from this one sample*, with replacement, and repeat to build the distribution of resampled means.

**House habit before any 10,000-rep loop: test it at 10 reps first.** If something's wrong with the logic, you want to find out in a second, not a minute.

In [ ]:
# Sanity check at small scale first
test_means = make_array()
for i in np.arange(10):
    test_sample = pop_sample.sample()
    test_means = np.append(test_means, np.mean(test_sample.column("Height")))
test_means

That ran fast and the numbers look like plausible heights. Now scale up.

In [ ]:
bootstrap_means = make_array()
for i in np.arange(10000):
    bootstrap_sample = pop_sample.sample()
    bootstrap_means = np.append(bootstrap_means, np.mean(bootstrap_sample.column("Height")))

Table().with_column("bootstrapped mean", bootstrap_means).hist(bins=30)

> **Handout Q1.2 (Student Challenge 1).** Find the 95% confidence interval for the sample mean from the distribution above.

In [ ]:
left = ...
right = ...
print(f"The 95% CI based on bootstrapping ranges from {left} to {right}")

> **Handout Q1.3 -- Predict first.** Now imagine we'd measured 500 people instead of 10 -- 50x as much data. Before you compute anything: will the new CI be narrower, wider, or about the same width? Roughly what factor do you expect (2x narrower? 10x? 50x)? Write your guess on the handout, *then* run the cells below.

In [ ]:
pop_sample_500 = pop_ht.sample(500)
pop_sample_500

> **Handout Q1.4 (Student Challenge 2).** Compute the bootstrapped 95% CI for this larger sample, the same way you did above.

In [ ]:
bootstrap_means_500 = make_array()
for i in np.arange(10000):
    bootstrap_sample = ...
    sample_avg = ...
    bootstrap_means_500 = np.append(bootstrap_means_500, sample_avg)

Table().with_column("bootstrapped mean", bootstrap_means_500).hist(bins=30)

In [ ]:
left_500 = ...
right_500 = ...
print(f"The 95% CI based on bootstrapping ranges from {left_500} to {right_500}")

> **Handout Q1.5 Discussion.** How close was your prediction in Q1.3? What does this tell you about the relationship between sample size and the width of a confidence interval?

## Part 2: From a Mean to a Slope

Last week (Class 18) you fit a regression line to 15 crickets: `temperature = 3.29 x chirps/sec + 25.23`, with r = 0.835.

That line came from only 15 crickets. If you'd measured a *different* 15 crickets, you would not get exactly the same slope.

> **Handout Q2.1 -- Predict first.** With only 15 data points, how much do you think the slope could plausibly shift if we measured a different 15 crickets? Give a rough range on the handout before running anything below.

In [ ]:
crickets = Table.read_table("./data/cricket_thermometer.csv")
crickets = crickets.relabeled("chirps_per_sec", "Chirps/sec").relabeled("temp_f", "Temp (F)")
crickets

Same regression toolbox you'll use again in Lab 09 -- `standard_units`, `correlation`, `slope`, `intercept` -- built from scratch rather than a black-box fitting call.

In [ ]:
def standard_units(x):
    "Convert an array of numbers to standard units."
    return (x - np.mean(x)) / np.std(x)

def correlation(t, label_x, label_y):
    x_su = standard_units(t.column(label_x))
    y_su = standard_units(t.column(label_y))
    return np.mean(x_su * y_su)

def slope_of(t):
    # Least-squares slope relating Temp (F) to Chirps/sec for table t.
    r = correlation(t, "Chirps/sec", "Temp (F)")
    return r * np.std(t.column("Temp (F)")) / np.std(t.column("Chirps/sec"))

def intercept_of(t):
    # Least-squares intercept relating Temp (F) to Chirps/sec for table t.
    return np.mean(t.column("Temp (F)")) - slope_of(t) * np.mean(t.column("Chirps/sec"))

observed_slope = slope_of(crickets)
observed_intercept = intercept_of(crickets)
print(f"Observed slope from all 15 crickets: {observed_slope:.4f}")
print(f"Observed intercept: {observed_intercept:.4f}")

Same pattern as Part 1: resample the table (with replacement), refit the slope, repeat. **Small-scale check first:**

In [ ]:
test_slopes = make_array()
for i in np.arange(10):
    resample = crickets.sample()
    test_slopes = np.append(test_slopes, slope_of(resample))
test_slopes

Now scale up to 10,000 resamples.

In [ ]:
boot_slopes = make_array()
for i in np.arange(10000):
    resample = crickets.sample()
    boot_slopes = np.append(boot_slopes, slope_of(resample))

Table().with_column("bootstrapped slope", boot_slopes).hist(bins=30)
left_slope = percentile(2.5, boot_slopes)
right_slope = percentile(97.5, boot_slopes)
print(f"The 95% CI for the slope ranges from {left_slope:.3f} to {right_slope:.3f}")

Here's what that uncertainty looks like directly on the scatter plot: each faint line is the regression line from one bootstrap resample.

In [ ]:
fig, ax = plt.subplots(figsize=(6,4.2))
xs = np.linspace(13.5, 21, 50)
x = crickets.column("Chirps/sec")
y = crickets.column("Temp (F)")

for i in range(40):
    resample = crickets.sample()
    s = slope_of(resample)
    b = intercept_of(resample)
    ax.plot(xs, s*xs + b, color="#9E1B34", alpha=0.10, linewidth=1)

ax.plot(xs, observed_slope*xs + observed_intercept, color="black", linewidth=2.5,
        label=f"Observed slope = {observed_slope:.2f}")
ax.scatter(x, y, color="black", zorder=5, s=25)
ax.set_xlabel("Chirps / second")
ax.set_ylabel("Temperature (F)")
ax.legend()
plt.show()

> **Handout Q2.2 Discussion.** Does the 95% CI for the slope include 0? What would it mean, scientifically, if it did? How does the width of this CI compare to the width you predicted in Q2.1?

> **Handout Q2.3.** Compare this CI to the one you found for the height mean in Part 1. Which sample gives the more precise estimate relative to its own scale: n = 15 crickets or n = 10 people? What's driving the difference?

### Adapt: bootstrap the intercept

The slope isn't the only quantity we might want a confidence interval for. You already have `intercept_of` from the toolbox above -- now write the resampling loop yourself, mirroring the slope loop.

> **Handout Q2.4 (Student Challenge 3).** Bootstrap a 95% CI for the intercept.

In [ ]:
boot_intercepts = make_array()
for i in np.arange(10000):
    resample = ...
    boot_intercepts = np.append(boot_intercepts, ...)

left_int = ...
right_int = ...
print(f"The 95% CI for the intercept ranges from {left_int} to {right_int}")

## Part 3: Looking Ahead

Lab 09 (Age of the Universe) asks you to do exactly this: bootstrap the slope of a regression line, on real supernova speed/distance data, then convert that slope into an estimate of the age of the universe (in billions of years), with a confidence interval. Everything you just did in Part 2 is the method you'll need there.

## Part 4: When Does This Break? (paper discussion, no code)

> **Handout Q4.1.** Bootstrapping assumes your one sample is a reasonable stand-in for the population. What has to be true about *how* a sample was collected for that assumption to hold? Can you think of a way the cricket sample or the height sample could have been collected that would make bootstrapping misleading?

> **Handout Q4.2.** Suppose a classmate ran only 200 bootstrap resamples (instead of 10,000) and got a *narrower* slope CI than yours. Should they trust their tighter interval more? Why or why not? (Hint: think back to the arbitrary lag-2 choice from Lab 7 -- a result isn't more trustworthy just because it's more convenient.)

> **Handout Q4.3.** With only 15 crickets, a bootstrap resample can, by chance, leave out an influential point entirely or include it two or three times. Why does that matter more for n = 15 than it would for n = 500?